<a href="https://colab.research.google.com/github/SPTKL/duckdb-geosupport/blob/main/notebooks/geosupport_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NYC geocoding with Geosupport and DuckDB

This walkthrough loads a prebuilt DuckDB extension, installs NYC's **26C / 26.3**
engine and data, and geocodes a small table. You'll inspect return codes, look up a
tax lot and building, and export results with their release provenance.

[Project](https://github.com/SPTKL/duckdb-geosupport) ·
[SQL reference](https://sptkl.github.io/duckdb-geosupport/)

Use a **Python 3, CPU runtime** in Google Colab (hardware accelerator: **None**).
Run the cells in order, or choose **Runtime → Run all**. The optional CSV upload,
26B comparison, and file download are off by default, so Run all needs no prompts.
The first setup takes several minutes and needs internet access and several GiB
of free disk. The 26C archive is about **195 MiB**, expanding to **1.9 GiB**.

Python hosts DuckDB and displays tables; the extension and Geosupport worker are
native C++. No compiler, Make, Git, Docker, or GitHub token is needed. This is an **unsigned development
extension**, so the connection explicitly allows unsigned extensions.

Colab's runtime files are temporary: download your exports before the session
ends. A saved notebook does not preserve the installed engine or data. See the
[Colab FAQ](https://research.google.com/colaboratory/faq.html) for runtime details.

## 1. Install DuckDB and download the extension

We use **DuckDB 1.4.4** and the **v0.2.0** Linux x86-64 release. The next cells
install the Python packages, check the Linux runtime libraries, and download a
prebuilt extension with a pinned SHA-256 checksum. **There is no source build.**
The archive also includes the project and third-party licenses.

The binary was built on Ubuntu 22.04 and tested on Ubuntu 22.04 and 24.04.
If the runtime libraries are already present, the system-package step does
nothing. Otherwise, it installs only libcurl, OpenSSL, and the C++ runtime.
The Geosupport engine and data download happens separately in section 2.

In [17]:
import ctypes
import hashlib
import io
import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path
from urllib.request import urlopen

assert platform.system() == "Linux" and platform.machine() == "x86_64", (
    "Use a Linux x86-64 Colab runtime for this notebook."
)
WORK = Path("/content")
WORK.mkdir(exist_ok=True)
free_gib = shutil.disk_usage(WORK).free / 1024**3
assert free_gib >= 4, f"Allow at least 4 GiB free for setup; available: {free_gib:.1f} GiB"
SOURCE_COMMIT = "c410b512826355c687b085a2cf2f346673f03313"
EXTENSION_RELEASE = "v0.2.0"
ASSET_URL = (
    "https://github.com/SPTKL/duckdb-geosupport/releases/download/"
    "v0.2.0/geosupport-0.2.0-Linux-x86_64.tar.gz"
)
ASSET_SHA256 = "e2157e513f8e47b6d9fc1a097f222f6520f14b171cf8a353a726e12eb54752f2"
STORE = WORK / "geosupport-store"
os.environ["GEOSUPPORT_HOME"] = str(STORE)
os.environ["GEOSUPPORT_RUNTIME"] = "local"
print(f"Runtime: {platform.system()} {platform.machine()} | Free disk: {free_gib:.1f} GiB")

Runtime: Linux x86_64 | Free disk: 85.2 GiB


In [18]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "duckdb==1.4.4", "pandas"], check=True)
missing_libraries = []
for library in ("libcurl.so.4", "libcrypto.so.3", "libstdc++.so.6"):
    try:
        ctypes.CDLL(library)
    except OSError:
        missing_libraries.append(library)
if missing_libraries:
    print("Installing missing runtime libraries:", ", ".join(missing_libraries))
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run([
        "apt-get", "install", "-y", "-qq", "--no-install-recommends",
        "libcurl4", "libssl3", "libstdc++6",
    ], check=True, env={**os.environ, "DEBIAN_FRONTEND": "noninteractive"})
else:
    print("Linux runtime libraries are already installed.")

Linux runtime libraries are already installed.


In [19]:
PACKAGE = WORK / f"geosupport-{EXTENSION_RELEASE}-linux-x86_64.tar.gz"
if not PACKAGE.is_file() or hashlib.sha256(PACKAGE.read_bytes()).hexdigest() != ASSET_SHA256:
    with urlopen(ASSET_URL, timeout=120) as response:
        archive_bytes = response.read()
    if hashlib.sha256(archive_bytes).hexdigest() != ASSET_SHA256:
        raise ValueError("Extension archive checksum mismatch; refusing to load it.")
    PACKAGE.write_bytes(archive_bytes)

# Read just the named binary; do not extract arbitrary archive paths.
EXTENSION = WORK / "geosupport-extension" / EXTENSION_RELEASE / "geosupport.duckdb_extension"
EXTENSION.parent.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(PACKAGE.read_bytes()), mode="r:gz") as package:
    member = package.getmember("geosupport.duckdb_extension")
    assert member.isfile(), "Expected a regular extension binary in the release archive."
    extension_bytes = package.extractfile(member).read()
temporary = EXTENSION.with_suffix(".tmp")
temporary.write_bytes(extension_bytes)
temporary.replace(EXTENSION)
print(f"Verified {EXTENSION_RELEASE} archive and installed: {EXTENSION}")

Verified v0.2.0 archive and installed: /content/geosupport-extension/v0.2.0/geosupport.duckdb_extension


## 2. Open DuckDB and install 26C

Set the store location **before the extension's first use**. It is separate from
any DuckDB database file. `gs_install` explicitly downloads and verifies NYC's
archive; repeating it verifies and reuses the installed bundle. Geocoding itself
never installs missing data.

`gs_load` starts the worker. Pin the connection to its immutable `bundle_id` so
subsequent queries use that exact engine/data bundle.

In [20]:
import duckdb
import pandas as pd
from IPython.display import display

assert duckdb.__version__ == "1.4.4", "Restart the runtime if a different DuckDB was already imported."
# Close an earlier notebook connection when this cell is rerun.
if "con" in globals():
    con.close()
con = duckdb.connect(config={"allow_unsigned_extensions": True})
con.execute("LOAD '" + str(EXTENSION).replace("'", "''") + "'")
display(con.sql("SELECT * FROM gs_available_releases()").df())

,release,bundle_id,engine_version,platform,details
0,26b,None,26.2,linux_amd64,"{""archive"":{""sha256"":""9a0db7df4911f14a4c974de3..."
1,26c,None,26.3,linux_amd64,"{""archive"":{""sha256"":""71a3f2d621348d3d70ed2829..."


In [21]:
RELEASE = "26c"
display(con.execute("CALL gs_install(?)", [RELEASE]).df())
con.execute("SET VARIABLE geosupport_release = (SELECT bundle_id FROM gs_load('26c'))")
BUNDLE_ID = con.sql("SELECT getvariable('geosupport_release')").fetchone()[0]
print("Pinned bundle:", BUNDLE_ID)
display(con.sql("SELECT * FROM gs_installed_releases()").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,release,bundle_id,engine_version,platform,details
0,26c,bce0275d8f061d45d4cf916647474cda66700544b0fc57...,26.3,linux_amd64,"{""adapter"":""native-c++"",""bundle_id"":""bce0275d8..."


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pinned bundle: bce0275d8f061d45d4cf916647474cda66700544b0fc57c930c30225b5979ecd


,release,bundle_id,engine_version,platform,details
0,26c,bce0275d8f061d45d4cf916647474cda66700544b0fc57...,26.3,linux_amd64,"{""adapter"":""native-c++"",""bundle_id"":""bce0275d8..."


## 3. Geocode one address

`gs_address(house_number, street_name, borough)` uses Geosupport function **1B**
and returns a typed SQL struct. Keep house numbers, BBLs, and BINs as strings.
For this 26C example, the expected BBL is `1001680032` and BIN is `1001831`.
Coordinates and identifiers may be NULL for other inputs.

In [22]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE first_result AS
    SELECT gs_address('125', 'Worth St', 'Mn') AS geo
""")
display(con.sql("""
    SELECT geo.bbl, geo.bin, geo.latitude, geo.longitude,
           geo.grc, geo.grc2, geo.release, geo.engine_version
    FROM first_result
""").df())
assert con.sql("SELECT geo.bbl FROM first_result").fetchone()[0] == "1001680032"

,bbl,bin,latitude,longitude,grc,grc2,release,engine_version
0,1001680032,1001831,40.715669,-74.002352,00,00,26c,26.3


## 4. Geocode a table

The sample includes three real addresses and an intentionally unrecognized street.
The fourth row demonstrates how to retain a rejected address for review.
Hyphenated Queens house numbers must remain strings.

In [23]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE addresses AS
    SELECT * FROM (VALUES
      (1, '125',    'Worth St',          'Mn'),
      (2, '350',    '5 Ave',             'Mn'),
      (3, '30-30',  'Thomson Ave',       'QN'),
      (4, '999999', 'NOT A REAL STREET', 'Mn')
    ) AS t(id, house_number, street_name, borough)
""")
display(con.sql("SELECT * FROM addresses ORDER BY id").df())

,id,house_number,street_name,borough
0,1,125,Worth St,Mn
1,2,350,5 Ave,Mn
2,3,30-30,Thomson Ave,QN
3,4,999999,NOT A REAL STREET,Mn


### Optional: replace the sample with your own CSV

Set `USE_UPLOADED_CSV = True` and run the next cell to select one CSV with headers
`id,house_number,street_name,borough`. Use borough codes `1`–`5` or aliases such as
`Mn` and `QN`. All columns are read as strings to preserve identifiers and house
numbers. Start with a small file: this extension serializes geocoding calls.

After an upload, run the geocoding and export cells again. To return to the
examples, rerun the sample-addresses cell above.

In [24]:
USE_UPLOADED_CSV = False
if USE_UPLOADED_CSV:
    import io
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select exactly one CSV file.")
    incoming = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))), dtype=str, keep_default_na=False)
    required = ["id", "house_number", "street_name", "borough"]
    missing = set(required) - set(incoming.columns)
    if missing:
        raise ValueError(f"Missing CSV columns: {sorted(missing)}")
    con.register("uploaded_addresses", incoming[required])
    con.execute("CREATE OR REPLACE TEMP TABLE addresses AS SELECT * FROM uploaded_addresses")
    con.unregister("uploaded_addresses")
    display(con.sql("SELECT * FROM addresses LIMIT 10").df())
else:
    print("Using the sample addresses.")

Using the sample addresses.


In [25]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE geocoded AS
    SELECT *, gs_address(house_number, street_name, borough) AS geo
    FROM addresses
""")
display(con.sql("""
    SELECT id, house_number, street_name, borough,
           geo.bbl, geo.bin, geo.latitude, geo.longitude, geo.grc, geo.grc2
    FROM geocoded ORDER BY id
""").df())

,id,house_number,street_name,borough,bbl,bin,latitude,longitude,grc,grc2
0,1,125,Worth St,Mn,1001680032,1001831,40.715669,-74.002352,00,00
1,2,350,5 Ave,Mn,1008350041,1015862,40.748452,-73.985712,00,00
2,3,30-30,Thomson Ave,QN,4002770001,4003533,40.744071,-73.936077,00,00
3,4,999999,NOT A REAL STREET,Mn,None,None,NaN,NaN,EE,EE


## 5. Review Geosupport return codes and messages

A rejected address stays in the result table. For address lookups, inspect both
`grc` / `message` and `grc2` / `message2`. The filter below flags anything other
than two exact `00` codes for review, including NULLs; your acceptance policy may
allow additional codes. It does not discard rows from the export.

The intentionally unrecognized street returns `EE` in both code fields with
messages and NULL identifiers/coordinates in the captured 26C example.
Invalid API inputs or runtime failures instead raise a query error.

In [26]:
review = con.sql("""
    SELECT id, house_number, street_name, borough,
           geo.grc, geo.message, geo.grc2, geo.message2
    FROM geocoded
    WHERE geo.grc IS DISTINCT FROM '00'
       OR geo.grc2 IS DISTINCT FROM '00'
    ORDER BY id
""").df()
with pd.option_context("display.max_colwidth", None):
    display(review)

,id,house_number,street_name,borough,grc,message,grc2,message2
0,4,999999,NOT A REAL STREET,Mn,EE,'NOT A REAL STREET' NOT RECOGNIZED. IS IT 'NOTRE DAME SCHOOL'?,EE,'NOT A REAL STREET' NOT RECOGNIZED. IS IT 'NOTRE DAME SCHOOL'?


## 6. Look up a tax lot, a building, or another function

`gs_bbl` uses function BL; `gs_bin` uses BN. An explicit `release` argument
overrides the connection default. `gs_call` accepts a JSON object of input fields
for other Geosupport functions. Each result's `raw` field contains the full parsed
output as JSON text. See the [function reference](https://sptkl.github.io/duckdb-geosupport/#functions)
for modes and available functions.

In [27]:
display(con.sql("""
    SELECT (gs_bbl('1001680032', release := '26c')).bbl AS tax_lot_bbl,
           (gs_bin('1001831', release := '26c', mode := 'tpad')).bbl AS building_bbl
""").df())

,tax_lot_bbl,building_bbl
0,1001680032,1001680032


In [28]:
generic = con.sql("""
    SELECT gs_call('1E', to_json({
      'house_number': '125',
      'street_name': 'Worth St',
      'borough_code': 'Mn'
    }), release := '26c', mode := 'extended') AS geo
""").fetchone()[0]
display({key: generic[key] for key in ("bbl", "bin", "grc", "message", "release")})
display(json.loads(generic["raw"]))

{'bbl': '1001680032',
 'bin': '1001831',
 'grc': '00',
 'message': None,
 'release': '26c'}

{'1990 Census Tract': '003100',
 '2000 Census Block': '1012',
 '2000 Census Block Suffix': '',
 '2000 Census Tract': '003100',
 '2010 Census Block': '1012',
 '2010 Census Block Suffix': '',
 '2010 Census Tract': '003100',
 '2020 Census Block': '1011',
 '2020 Census Block Suffix': '',
 '2020 Census Tract': '003100',
 '2020 Community District Tabulation Area (CDTA)': 'MN01',
 '2020 Neighborhood Tabulation Area (NTA)': 'MN0102',
 'Alley or Cross Street List Flag': '',
 'Angle to From Node - Beta Value': '',
 'Angle to To Node - Alpha Value': '',
 'Assembly District': '65',
 'Atomic Polygon': '112',
 'B10SC - First Borough and Street Code': '14549001010',
 'B10SC - Second Borough and Street Code': '',
 'B10SC - Third Borough and Street Code': '',
 'BOE LGC Pointer': '1',
 'BOE Preferred B7SC': '14549001',
 'BOE Preferred Street Name': 'WORTH STREET',
 'BOROUGH BLOCK LOT (BBL)': {'BOROUGH BLOCK LOT (BBL)': '1001680032',
  'Borough Code': '1',
  'Tax Block': '00168',
  'Tax Lot': '0032'},
 '

## 7. Optional: compare quarters

Set `COMPARE_26B = True` to install another quarter and compare the same address.
This downloads another archive and needs roughly **2 GiB more disk**. Differences
are not guaranteed for a particular address. The connection remains pinned to
26C; each comparison query explicitly selects its release.

In [29]:
COMPARE_26B = False
if COMPARE_26B:
    display(con.sql("CALL gs_install('26b')").df())
    display(con.sql("""
        WITH comparison AS (
          SELECT '26b' AS quarter, gs_address('125', 'Worth St', 'Mn', release := '26b') AS geo
          UNION ALL
          SELECT '26c', gs_address('125', 'Worth St', 'Mn', release := '26c')
        )
        SELECT quarter, geo.bbl, geo.bin, geo.latitude, geo.longitude,
               geo.grc, geo.grc2, geo.engine_version, geo.bundle_id
        FROM comparison ORDER BY quarter
    """).df())
else:
    print("Skipping the optional 26B download.")

Skipping the optional 26B download.


## 8. Export results and keep their provenance

Export all rows, including rejects, to CSV and Parquet. Both files include the
original input, return codes/messages, raw JSON, and release identity. The JSON
sidecar records the extension source revision, release archive checksum, and binary hash.

Files are written under `/content/geosupport-results` and replaced when this cell
is rerun. Download the ZIP through Colab's Files panel, or enable the next cell.
The ZIP contains results and provenance, **not** the installed engine/data store.
For historical reproducibility, retain the matching Geosupport bundle separately;
the bundle ID alone is not a backup.

In [30]:
OUTPUT = WORK / "geosupport-results"
OUTPUT.mkdir(exist_ok=True)
con.execute("""
    CREATE OR REPLACE TEMP VIEW export_results AS
    SELECT id, house_number, street_name, borough,
           geo.bbl AS bbl, geo.bin AS bin,
           geo.latitude AS latitude, geo.longitude AS longitude,
           geo.grc AS grc, geo.grc2 AS grc2,
           geo.message AS message, geo.message2 AS message2,
           geo.release AS release, geo.engine_version AS engine_version,
           geo.bundle_id AS bundle_id, geo.raw AS raw
    FROM geocoded
""")
for name, options in [("geocoded.csv", "FORMAT CSV, HEADER true"), ("geocoded.parquet", "FORMAT PARQUET")]:
    destination = str(OUTPUT / name).replace("'", "''")
    con.execute(f"COPY (SELECT * FROM export_results ORDER BY id) TO '{destination}' ({options})")
provenance = {
    "repository": "https://github.com/SPTKL/duckdb-geosupport",
    "source_commit": SOURCE_COMMIT,
    "extension_release": EXTENSION_RELEASE,
    "extension_archive_url": ASSET_URL,
    "extension_archive_sha256": ASSET_SHA256,
    "duckdb_version": duckdb.__version__,
    "extension_sha256": hashlib.sha256(EXTENSION.read_bytes()).hexdigest(),
    "release": RELEASE,
    "bundle_id": BUNDLE_ID,
    "rows": con.sql("SELECT count(*) FROM geocoded").fetchone()[0],
}
(OUTPUT / "provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")
archive = shutil.make_archive(str(WORK / "geosupport-results"), "zip", OUTPUT)
display(provenance)
print("Results ZIP:", archive)

{'repository': 'https://github.com/SPTKL/duckdb-geosupport',
 'source_commit': 'c410b512826355c687b085a2cf2f346673f03313',
 'extension_release': 'v0.2.0',
 'extension_archive_url': 'https://github.com/SPTKL/duckdb-geosupport/releases/download/v0.2.0/geosupport-0.2.0-Linux-x86_64.tar.gz',
 'extension_archive_sha256': 'e2157e513f8e47b6d9fc1a097f222f6520f14b171cf8a353a726e12eb54752f2',
 'duckdb_version': '1.4.4',
 'extension_sha256': '43b52a2bdee3d3e4449b4ba05e8c58ee020e41f31b8d4bb499e64d75515602b7',
 'release': '26c',
 'bundle_id': 'bce0275d8f061d45d4cf916647474cda66700544b0fc57c930c30225b5979ecd',
 'rows': 4}

Results ZIP: /content/geosupport-results.zip


In [31]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive)
else:
    print("Download /content/geosupport-results.zip from Colab's Files panel.")

Download /content/geosupport-results.zip from Colab's Files panel.


## Next steps and rerunning

- Upload your CSV in section 4, then rerun geocoding, review, and export.
- If setup was interrupted, rerun the failed cell before continuing. If DuckDB was
  already imported at a different version, restart the session and rerun from the top.
- If the runtime was deleted, run setup again; a notebook saved to Drive does not
  include its VM files or the Geosupport store.
- Keep `release`, `engine_version`, and `bundle_id` with every result. Consult the
  [SQL reference](https://sptkl.github.io/duckdb-geosupport/) and
  [repository README](https://github.com/SPTKL/duckdb-geosupport#readme) for local use.

The adapter is MIT licensed. NYC engine/data terms apply separately; see
[third-party notices](https://github.com/SPTKL/duckdb-geosupport/blob/main/THIRD_PARTY_NOTICES.md).